In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

153

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
import json

user_prompt = json.dumps(doc)

In [9]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [10]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [11]:
result = response.output_parsed

print(result)

questions=['I just found this course — am I still able to join now?', 'If I start late, can I still take the course even though it already began?', 'Can someone who discovered the course after it started still participate?', 'Is it too late to join, or can I still sign up now?', 'If I join after the course has already started, can I still get the certificate?']


In [12]:
print(result.questions)

['I just found this course — am I still able to join now?', 'If I start late, can I still take the course even though it already began?', 'Can someone who discovered the course after it started still participate?', 'Is it too late to join, or can I still sign up now?', 'If I join after the course has already started, can I still get the certificate?']


In [13]:
from evaluation_utils import llm_structured

In [14]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — is it too late to join now?', 'Can I still start the course after it’s already begun?', 'If I join late, do I still have a shot at getting the certificate?', 'What’s the deadline for the project if I want the certificate?', 'Am I allowed to join the course now, and what do I need to do to qualify for a certificate?']


In [15]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — is it too late to join now?',
  'document': '74eb249bbf'},
 {'question': 'Can I still start the course after it’s already begun?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, do I still have a shot at getting the certificate?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for the project if I want the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to join the course now, and what do I need to do to qualify for a certificate?',
  'document': '74eb249bbf'}]

In [17]:
from evaluation_utils import llm_structured_retry

In [18]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [19]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [22]:
len(documents)

153

In [23]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [24]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/153 [00:00<?, ?it/s]

In [25]:
results

[([{'question': 'I found this course late — can I still enroll and participate, or is it too late now?',
    'document': '74eb249bbf'},
   {'question': 'If I join the course after it already started, am I still eligible for a certificate?',
    'document': '74eb249bbf'},
   {'question': 'Do latecomers get to take part in the course, or are registrations closed completely?',
    'document': '74eb249bbf'},
   {'question': 'What’s the cutoff for getting a certificate if I’m just discovering the course now?',
    'document': '74eb249bbf'},
   {'question': 'Can I still join the course after it has begun, and what do I need to do to get certified?',
    'document': '74eb249bbf'}],
  ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=111, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=318)),
 ([{'question': 'Do I actually need a confirmation email to start the LLM Zoomcamp, or am I already 

In [2]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

NameError: name 'results' is not defined

In [27]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.12165974999999996

In [28]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.12165974999999996

In [29]:
ground_truth

[{'question': 'I found this course late — can I still enroll and participate, or is it too late now?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course after it already started, am I still eligible for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do latecomers get to take part in the course, or are registrations closed completely?',
  'document': '74eb249bbf'},
 {'question': 'What’s the cutoff for getting a certificate if I’m just discovering the course now?',
  'document': '74eb249bbf'},
 {'question': 'Can I still join the course after it has begun, and what do I need to do to get certified?',
  'document': '74eb249bbf'},
 {'question': 'Do I actually need a confirmation email to start the LLM Zoomcamp, or am I already in?',
  'document': '977bf7786c'},
 {'question': 'If I signed up for the course, should I still wait for an acceptance email before watching the lessons or doing homework?',
  'document': '977bf7786c'},
 {'question': 'Is there any list tha

In [1]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

NameError: name 'ground_truth' is not defined

In [32]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)